# 03b – World-wide check of the prototype

Runs the web app's own code (`prototype/app.py`) for **17 requests**: 11 land places on all continents with a period
that has Sentinel-1 data, 3 of these places again in 2023 (when Sentinel-1 did not record them, see below), and 3 places
where maps are impossible (open sea, Svalbard, an area crossing the 180° meridian). Small 4 km areas are used.

After Sentinel-1B failed in December 2021, only Sentinel-1A recorded until Sentinel-1C started in 2025, and many regions
were recorded less often or not at all. The first run of this check found three places without any IW recording in
2023 (Lusaka, the Algerian Sahara, Ulaanbaatar); they are kept as examples where the app must give a clear message.

Every place is sorted into one outcome:

| outcome | meaning |
|---|---|
| complete | 12 monthly maps |
| partial | 1–11 monthly maps: some months have no Sentinel-1 image on the chosen track (reported with the months) |
| unavailable | clear message: no Sentinel-1 images for this place and period |
| rejected | clear message: place not supported (polar latitude, 180° meridian) |
| error | any other message (failed download, model problem) – needs a look |
| crash | unexpected Python error – a bug |

For every place with maps, the tables are checked automatically: 12 consecutive months, class shares adding up to
100 %, and month-to-month / total changes equal to the differences of the shares (in percentage points).
The prediction time of each mode is measured.

Both prediction modes are checked (standard and ensemble). The cell stops with an error if any request does not behave as expected. This check tests the software and its error handling, not the classification accuracy (that is measured in 02c and 02e). Run this only after the final model is saved in
`models/round3` (notebooks 02d and 02c). Clear the output of the login cell before sharing.

In [1]:
import sys
import subprocess
import importlib
from pathlib import Path

# prototype packages (Earth Engine, rasterio, gradio, ...) from requirements.txt
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
importlib.invalidate_caches()          # lets Python find packages that were installed just now

for module, package in [("ee", "earthengine-api"), ("rasterio", "rasterio"), ("gradio", "gradio>=5")]:
    try:
        importlib.import_module(module)
    except ModuleNotFoundError:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        importlib.invalidate_caches()
        importlib.import_module(module)
print("packages ready")

packages ready


In [2]:
import tensorflow as tf

# take GPU memory only when it is needed, so this notebook also runs while another notebook holds some of it
# (must run before the first model is loaded; after a kernel restart this cell has to come first)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPUs:", tf.config.list_physical_devices("GPU"))
except RuntimeError:
    print("the GPU was already in use in this kernel - restart the kernel and run all cells from the top")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import sys
import subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

# keras-hub (for SegFormer) is installed WITHOUT its dependencies: a normal install
# pulls in another TensorFlow version and breaks the existing one.
# The keras-hub version must fit the installed Keras: 0.33.0 needs Keras >= 3.15,
# 0.24.0 works with Keras 3.8 - 3.14 (tested with TensorFlow 2.20 + Keras 3.11.2).
from importlib.metadata import version as package_version, PackageNotFoundError

keras_version = tuple(int(part) for part in package_version("keras").split(".")[:2])
KERAS_HUB = "0.33.0" if keras_version >= (3, 15) else "0.24.0"

try:
    installed = package_version("keras-hub")
except PackageNotFoundError:
    installed = None

if installed != KERAS_HUB:
    print(f"installing keras-hub {KERAS_HUB} (found: {installed}) for Keras {package_version('keras')}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall",
                    f"keras-hub=={KERAS_HUB}"], check=True)
    if "keras_hub" in sys.modules:
        raise RuntimeError("keras-hub was replaced - restart the kernel and run this cell again")

# small helpers keras-hub needs: downloads from Kaggle Models / Hugging Face (no TensorFlow dependency)
missing = []
for package in ["kagglehub", "huggingface_hub", "regex", "rich"]:
    try:
        package_version(package.replace("_", "-"))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    print("installing", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import keras_hub
print("keras-hub:", keras_hub.__version__)

installing keras-hub 0.24.0 (found: None) for Keras 3.11.2


installing ['kagglehub']


keras-hub: 0.24.0


In [4]:
import ee

PROJECT_ID = "sentinel1-lulc-project-512618"

# Only asks for a new login when the saved token is missing. Clear this cell's output before sharing.
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready:", ee.Number(1).getInfo())


Successfully saved authorization token.
Earth Engine ready: 1


In [5]:
sys.path.insert(0, "prototype")
import world_check

# standard mode: the selected model (what the app uses by default)
sys.argv = ["world_check.py", "--model-dir", "models/round3", "--project", PROJECT_ID, "--mode", "standard"]
assert world_check.main(), "some requests did not behave as expected (see the lines above)"

mode: standard | model folder: models/round3 | area: 4 km

Tokyo             lat   35.6800 lon   139.7600 from 2023-01 ...


   -> complete: 12/12 months; main class Built-up (mean share 82.3 %); common radar area 16.4 km² (100.0 % of the image)  (24 s)
Lusaka            lat  -15.4200 lon    28.2800 from 2021-03 ...
   -> partial: 10/12 months; main class Built-up (mean share 90.3 %); common radar area 16.1 km² (100.0 % of the image); no image in 2022-01, 2022-02  (3 s)
Reykjavik         lat   64.1500 lon   -21.9400 from 2022-06 ...
   -> complete: 12/12 months; main class Permanent water (mean share 56.2 %); common radar area 16.5 km² (100.0 % of the image)  (5 s)
Ushuaia           lat  -54.8000 lon   -68.3000 from 2023-01 ...
   -> complete: 12/12 months; main class Trees (incl. mangroves) (mean share 38.0 %); common radar area 16.4 km² (100.0 % of the image)  (4 s)
Sahara_Algeria    lat   25.0000 lon     5.0000 from 2021-01 ...
   -> complete: 12/12 months; main class Bare / sparse vegetation (mean share 76.4 %); common radar area 16.4 km² (100.0 % of the image)  (3 s)
Amazon_Rondonia   lat   -8.0000 lon 

In [6]:
# ensemble mode: the setting chosen in 02c (4 models x 8 flips/rotations)
sys.argv = ["world_check.py", "--model-dir", "models/round3", "--project", PROJECT_ID, "--mode", "ensemble"]
assert world_check.main(), "some requests did not behave as expected (see the lines above)"

mode: ensemble | model folder: models/round3 | area: 4 km

Tokyo             lat   35.6800 lon   139.7600 from 2023-01 ...


   -> complete: 12/12 months; main class Built-up (mean share 85.3 %); common radar area 16.4 km² (100.0 % of the image)  (44 s)
Lusaka            lat  -15.4200 lon    28.2800 from 2021-03 ...
   -> partial: 10/12 months; main class Built-up (mean share 92.3 %); common radar area 16.1 km² (100.0 % of the image); no image in 2022-01, 2022-02  (8 s)
Reykjavik         lat   64.1500 lon   -21.9400 from 2022-06 ...
   -> complete: 12/12 months; main class Permanent water (mean share 55.8 %); common radar area 16.5 km² (100.0 % of the image)  (8 s)
Ushuaia           lat  -54.8000 lon   -68.3000 from 2023-01 ...
   -> complete: 12/12 months; main class Trees (incl. mangroves) (mean share 36.2 %); common radar area 16.4 km² (100.0 % of the image)  (10 s)
Sahara_Algeria    lat   25.0000 lon     5.0000 from 2021-01 ...
   -> complete: 12/12 months; main class Bare / sparse vegetation (mean share 87.8 %); common radar area 16.4 km² (100.0 % of the image)  (8 s)
Amazon_Rondonia   lat   -8.0000 lon